<a href="https://colab.research.google.com/github/expely/Business-Analytics-Labs/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [54]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [55]:
# Read csv
df = pd.read_csv("/content/cleaned_airbnb_data_7.csv")
print(df.head())
print(df.shape)

      id                         listing_url       scrape_id last_scraped  \
0   3781   https://www.airbnb.com/rooms/3781  20250923202714   2025-09-23   
1   5506   https://www.airbnb.com/rooms/5506  20250923202714   2025-09-24   
2   6695   https://www.airbnb.com/rooms/6695  20250923202714   2025-09-24   
3   8789   https://www.airbnb.com/rooms/8789  20250923202714   2025-09-24   
4  10811  https://www.airbnb.com/rooms/10811  20250923202714   2025-09-24   

        source                                               name  \
0  city scrape                          HARBORSIDE-Walk to subway   
1  city scrape     ** Fort Hill Inn Private! Minutes to center!**   
2  city scrape      Fort Hill Inn *Sunny* 1 bedroom, condo duplex   
3  city scrape                Curved Glass Studio/1bd facing Park   
4  city scrape  Bostons Best Rentals -Studio Prestigious Back Bay   

                                         description  \
0  Fully separate apartment in a two apartment bu...   
1  **THE B

### ✍️ Your Response: 🔧
1. This includes all the information about different airbnb listings, where a ot of the variables are scaled to be between -1 - 1.

2. There are 4419 rows, and 163 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [56]:
columns_to_drop = [
    # 1. Identifiers and Scrape Metadata
    'id',
    'scrape_id',
    'last_scraped',
    'source',
    'host_id',

    # 2. URLs
    'listing_url',
    'host_url',
    'picture_url',

    # 3. Free-form text (Drop unless you are applying NLP/TF-IDF)
    'name',
    'description',

    # 5. Data Leakage (Derived from the target variable 'price')
    'price_scaled.1',
    'price_per_person.1',
    'price_scaled',
    'price_per_person',
    'estimated_revenue_l365d'
]
duplicate_cols = [col for col in df.columns if col.endswith('.1')]
columns_to_drop.extend(duplicate_cols)
df_clean = df.drop(columns=columns_to_drop, errors='ignore')

df_clean.dropna(inplace=True, subset=['price'])

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3506 entries, 0 to 4418
Data columns (total 71 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_name                                     3505 non-null   object 
 1   host_since                                    3505 non-null   object 
 2   host_location                                 2829 non-null   object 
 3   host_about                                    2408 non-null   object 
 4   host_response_time                            3363 non-null   object 
 5   host_response_rate                            3363 non-null   object 
 6   host_acceptance_rate                          3364 non-null   object 
 7   host_is_superhost                             3303 non-null   object 
 8   host_thumbnail_url                            3505 non-null   object 
 9   host_picture_url                              3505 non-null   object

### ✍️ Your Response: 🔧
1. I dropped any identifiers, as they're not real data, it's just for database normalization purposes. I also dropped URLs and any free-form text as they're not easily able to predict a number like price. I got rid of duplicate columns that didn't need to be there, and data leakages since they are based on price.

2. If we kept these in, the URLs and free-form text would likely give errors as the model doesn't know how to predict price from unstructured data. IDs could create relationships that don't exist, and data leakages/duplicate columns could over-influence the model created a biased model.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [57]:
# Only numbered columns
numeric_df = df_clean.select_dtypes(include=['number'])

# Create Correlation matrix
price_correlations = numeric_df.corr()['price'].sort_values(ascending=False)
print(price_correlations)

price                                           1.000000
host_total_listings_count                       0.282461
accommodates                                    0.149618
host_listings_count                             0.134423
maximum_nights                                  0.123481
bathrooms                                       0.109681
availability_60                                 0.104552
availability_30                                 0.102668
availability_90                                 0.098495
availability_eoy                                0.096857
availability_365                                0.081072
beds                                            0.077857
longitude                                       0.059468
bedrooms                                        0.046846
review_scores_location                          0.041168
review_scores_cleanliness                       0.023263
latitude                                        0.019307
review_scores_rating           

### ✍️ Your Response: 🔧
1. Price obviously has the highest correlation with itself, but that's not really included. The host_total_listings_count is the most positively correlated, probably because it means that host does this as their main source of income so they're likely in it for more money. The most negatively correlated is minimum_maximum_nights, so the lower the maximum nights someone can spend at a place the higher price is.

2. I think anything +-0.10 is going to be very useful, as we have quite a few variables with that. We could go for even more, but I think this current list of numeric attributes will all be useful. We'll have to check against the MSE though.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [58]:
# Separate the features and the target
numeric_df.dropna(inplace=True)
X = numeric_df.drop(columns=['price'])
y = numeric_df['price']
print(y.value_counts()[y.value_counts() == 1])

price
545.0     1
592.0     1
566.0     1
677.0     1
941.0     1
         ..
701.0     1
2238.0    1
612.0     1
313.0     1
284.0     1
Name: count, Length: 177, dtype: int64


### ✍️ Your Response: 🔧
1. For now I need a baseline, so I'm going to include all of the features, afterwards I can take the top 10 correlated features to see if that improves the model's performance.

2. This is a regression problem because we have a target variable that is a number. We're trying to predict a specific, measurable value, price.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [59]:
# Use features in X, and original target y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Check the shapes of the splits
X_train.shape, X_test.shape

((2220, 40), (555, 40))

## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [60]:
# Initialize and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_linear = linear_model.predict(X_test)

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [61]:
# Evaluate the model
mse = mean_squared_error(y_test, y_pred_linear)
r2 = r2_score(y_test, y_pred_linear)

# Print the evaluation metrics
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Variance: {y_test.var()}")
print(f"R-squared (R2): {r2:.2f}")

Mean Squared Error (MSE): 106633.94
Variance: 13371569.479090624
R-squared (R2): 0.99


### ✍️ Your Response: 🔧
1. The R squared is almost 100%, which is really really good. It means it's almost perfect... too perfect. This could be a serious case of overfitting.

2. The MSE is not too large or small. It looks like a healthy amount with how large the variance is. The MSE is less than the variance, but not too much smaller.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [62]:
# Display coefficients in order of highest to lowest correlation
coefficients_linear = pd.Series(linear_model.coef_, index=X.columns)
print("\nLinear Regression Coefficients (ordered):")
print(coefficients_linear.sort_values(ascending=False))


Linear Regression Coefficients (ordered):
longitude                                       1.383788e+03
calculated_host_listings_count                  1.076553e+03
review_scores_rating                            1.748563e+02
review_scores_accuracy                          5.745678e+01
review_scores_location                          5.508831e+01
bathrooms                                       5.497882e+01
bedrooms                                        5.293699e+01
review_scores_value                             3.614803e+01
accommodates                                    3.110196e+01
availability_60                                 5.471205e+00
availability_30                                 3.667734e+00
number_of_reviews_l30d                          2.746886e+00
availability_eoy                                1.768771e+00
maximum_minimum_nights                          1.298783e+00
host_listings_count                             4.540098e-01
number_of_reviews_ly                      

### ✍️ Your Response: 🔧
1. The largest increase was longitude surprisingly. It seems like the east side is a lot more expensive then the west. It does go back to the 3 rules of real estate! The other one is the host listings count, the more they have the higher the price was.

2. The negatively impacted was the number of shared, private, and entire homes which meant the more they had of them, the lower the price was. Which makes sense, considering if you're renting a private or shared room it'll be pretty low price.

3. One of the largest business insights is the 3 rules of real estate, location location location. As that was a really good indicator here with latitude being negatively correlated and longitude being positively correlated. We can also look at the hosts who have a lot of listings and start getting insights to help them adjust their price to the market rates. Higher ratings also meant higher price, so we can incentivize people to start rating more often, and incentivize higher ratings for the hosts.


## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [63]:
# Get top 5 features based on absolute coefficient magnitude
top_features = coefficients_linear.abs().sort_values(ascending=False).head(5)

X = X[top_features.keys()]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Rebuild regression model
linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

# Compare MSE and R^2
mse_new = mean_squared_error(y_test, y_pred_linear)
r2_new = r2_score(y_test, y_pred_linear)
print(f"Mean Squared Error (MSE): {mse_new:.2f} Old: {mse:.2f}")
print(f"R-squared (R2): {r2_new:.2f} Old: {r2:.2f}")

Mean Squared Error (MSE): 111099.20 Old: 106633.94
R-squared (R2): 0.99 Old: 0.99


### ✍️ Your Response: 🔧
1. I decided to keep the top 5, as we had 5 features that were +-10^3, so I didn't feel like I could just pick 3.

2. I feel like the model improved a little bit, the MSE went up, but given how high the variance is the MSE has the room to go up. The R^2 stayed the same as the old value though, being 99%.

3. I would probably reccommend this one, as there are a lot fewer variables to explain, so we can give a better detailed report to stakeholders. It's a lot easier to explain longitude with a map of the area, and the number of listings.

4. This relates directly, I'm doing the linear regression models that I created in excel, in a fraction of the time. It's really cool to me that I can write one line of code and that's all I have to do to create these models. In Excel it is a lot larger process to try and one-hot-encode and then scale, and then create the regression models compared to this. And it has a lot more customizability.


## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. How can hosts more accurately set a fair price for their listing?

2. If a host is going to buy a lot of properties, they should do it on the east side of town, the prices are a lot higher, and so the hosts can make a lot more revenue. When a host rents out just a shared or private room, it should be a lot cheaper then if they're renting the whole property.

3. Because the top 5 values really overshadowed the rest (despite not directly being related to price, surprisingly), it would be interesting to exclude them from the models so we can micro-analyze other features like bedrooms and what not to give more specific recommendations to hosts.

4. Answered in part 9.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [64]:
!jupyter nbconvert --to html "assignment_11_regression.ipynb"

[NbConvertApp] Converting notebook assignment_11_regression.ipynb to html
[NbConvertApp] Writing 339344 bytes to assignment_11_regression.html
